In [1]:
!pip install -q "trl==1.4.0" peft bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.6 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch, os, json, gc
import numpy as np, pandas as pd
from transformers import (AutoModelForCausalLM, AutoTokenizer,
                          BitsAndBytesConfig, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

LABEL_NAMES = {0:"Yardım Talebi",1:"Kayıp Bildirimi",2:"Altyapı Hasarı",
               3:"Bağış/Koordinasyon",4:"Diğer/İlgisiz"}
MODELS = [
    ("smollm2",   "SmolLM2-360M",   "HuggingFaceTB/SmolLM2-360M-Instruct"),
    ("tinyllama", "TinyLlama-1.1B", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"),
    ("qwen25",    "Qwen2.5-1.5B",   "Qwen/Qwen2.5-1.5B-Instruct"),
]
SEEDS = [123, 2023]
base = '/content/drive/MyDrive/bdm_proje'
out_dir = f'{base}/results/seeds'; os.makedirs(out_dir, exist_ok=True)
train_df = pd.read_csv(f'{base}/data/processed/train.csv')
val_df   = pd.read_csv(f'{base}/data/processed/val.csv')
test_df  = pd.read_csv(f'{base}/data/processed/test.csv')

def make_instruction(tw):
    return ("Sen bir deprem acil durum sınıflandırma asistanısın. "
            "Verilen tweeti aşağıdaki kategorilerden birine koy. "
            "SADECE kategori adını yaz, başka hiçbir şey yazma.\n\n"
            f"Tweet: {tw}\n\nKategoriler:\n- Yardım Talebi\n- Kayıp Bildirimi\n"
            "- Altyapı Hasarı\n- Bağış/Koordinasyon\n- Diğer/İlgisiz")

def parse_label(text):
    t = text.lower()
    if "yardım" in t or "yardim" in t:                      return 0
    if "kayıp" in t or "kayip" in t:                        return 1
    if "altyapı" in t or "altyapi" in t:                    return 2
    if "bağış" in t or "bagis" in t or "koordinasyon" in t: return 3
    if "diğer" in t or "diger" in t or "ilgisiz" in t:      return 4
    return -1

def to_pc(df, tok):
    rows=[]
    for _,r in df.iterrows():
        msgs=[{"role":"user","content":make_instruction(r['tweet'])}]
        p=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        rows.append({"prompt":p,"completion":LABEL_NAMES[int(r['label_id'])]})
    return Dataset.from_list(rows)

for KEY, NAME, MID in MODELS:
    for SEED in SEEDS:
        rp = f'{out_dir}/{KEY}_seed{SEED}_sonuc.json'
        if os.path.exists(rp):
            print(f"⏭️  {NAME} seed {SEED} zaten var — atlanıyor."); continue
        print(f"\n{'='*55}\n{NAME} — SEED {SEED} eğitiliyor...\n{'='*55}")
        torch.manual_seed(SEED); np.random.seed(SEED)
        tok = AutoTokenizer.from_pretrained(MID)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        tok.padding_side = "right"
        tr, vl = to_pc(train_df, tok), to_pc(val_df, tok)
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
              bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        mdl = AutoModelForCausalLM.from_pretrained(MID, quantization_config=bnb,
              device_map="auto", dtype=torch.bfloat16)
        mdl.config.use_cache=False
        mdl = prepare_model_for_kbit_training(mdl, use_gradient_checkpointing=True)
        mdl = get_peft_model(mdl, LoraConfig(r=16, lora_alpha=32,
              target_modules=["q_proj","v_proj","k_proj","o_proj"],
              lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM))
        cfg = SFTConfig(output_dir=f"/content/tmp_{KEY}_{SEED}", num_train_epochs=3,
              per_device_train_batch_size=4, per_device_eval_batch_size=4,
              gradient_accumulation_steps=4, learning_rate=2e-4, weight_decay=0.01,
              warmup_ratio=0.03, lr_scheduler_type="cosine", eval_strategy="steps",
              eval_steps=50, save_strategy="steps", save_steps=50, save_total_limit=1,
              load_best_model_at_end=True, metric_for_best_model="eval_loss",
              logging_steps=25, report_to="none", completion_only_loss=True, seed=SEED,
              fp16=False, bf16=True)
        tr_ = SFTTrainer(model=mdl, train_dataset=tr, eval_dataset=vl, args=cfg,
              processing_class=tok, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
        tr_.train()
        mdl.config.use_cache=True; mdl.eval()
        preds, trues = [], []
        for _, row in test_df.iterrows():
            msgs=[{"role":"user","content":make_instruction(row['tweet'])}]
            p=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            inp=tok(p, return_tensors="pt").to(mdl.device); n=inp['input_ids'].shape[1]
            with torch.no_grad():
                o=mdl.generate(**inp, max_new_tokens=12, do_sample=False, pad_token_id=tok.pad_token_id)
            g=parse_label(tok.decode(o[0][n:], skip_special_tokens=True).strip())
            preds.append(g if g!=-1 else 4); trues.append(int(row['label_id']))
        acc=accuracy_score(trues,preds)
        mf1=f1_score(trues,preds,average='macro',zero_division=0)
        wf1=f1_score(trues,preds,average='weighted',zero_division=0)
        json.dump({"model":NAME,"seed":SEED,"accuracy":round(float(acc),4),
                   "f1_macro":round(float(mf1),4),"f1_weighted":round(float(wf1),4)},
                  open(rp,'w',encoding='utf-8'), ensure_ascii=False, indent=2)
        print(f"✅ {NAME} seed {SEED}: Acc {acc:.4f} · Macro-F1 {mf1:.4f} → kaydedildi")
        del mdl, tr_, tok; gc.collect(); torch.cuda.empty_cache()

# ---- ÖZET: seed 42 (mevcut) + 123 + 2023 → ortalama ± std ----
print(f"\n\n{'='*55}\nÖZET — Ortalama ± Std (mevcut seed'ler)\n{'='*55}")
for KEY, NAME, _ in MODELS:
    accs, mf1s = [], []
    f42 = f'{base}/results/fine_tuned/{KEY}_qlora_sonuc.json'
    if os.path.exists(f42):
        d=json.load(open(f42)); accs.append(d['accuracy']); mf1s.append(d['f1_macro'])
    for SEED in SEEDS:
        fp=f'{out_dir}/{KEY}_seed{SEED}_sonuc.json'
        if os.path.exists(fp):
            d=json.load(open(fp)); accs.append(d['accuracy']); mf1s.append(d['f1_macro'])
    if accs:
        print(f"{NAME:16s} | n={len(accs)} seed | "
              f"Acc {np.mean(accs):.3f}±{np.std(accs):.3f} | "
              f"Macro-F1 {np.mean(mf1s):.3f}±{np.std(mf1s):.3f}")


SmolLM2-360M — SEED 123 eğitiliyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/633 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
